# Symptom-to-Doctor Matcher

**Goal:** Patient types symptoms in natural language → get ranked doctor recommendations.

**Approach:** Zero-shot semantic search
1. Embed each doctor's text profile using a local sentence-transformer model
2. Store embeddings in a FAISS index for fast retrieval
3. At query time: embed the patient's symptom input → find nearest doctor embeddings → return ranked results

**Model:** `all-MiniLM-L6-v2`

---

## Imports & config

In [40]:
import pandas as pd
import numpy as np
import faiss
import os
import time
from sentence_transformers import SentenceTransformer


DATA_PATH       = 'cleaning\cleaned_data.csv'   
INDEX_PATH      = 'vezeeta_faiss.index'
MODEL_NAME      = 'all-MiniLM-L6-v2'  
TOP_K           = 10                   # how many doctors to return per query
EMBEDDING_BATCH = 256                  


<>:9: SyntaxWarning: invalid escape sequence '\c'
<>:9: SyntaxWarning: invalid escape sequence '\c'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25620\2468042623.py:9: SyntaxWarning: invalid escape sequence '\c'
  DATA_PATH       = 'cleaning\cleaned_data.csv'


## Load & preprocess data

In [4]:
df = pd.read_csv(DATA_PATH, encoding='latin-1')
print(f'Loaded {len(df):,} doctors across {df["Speciality"].nunique()} specialties.')

# Fill nulls in text fields
text_cols = ['Speciality', 'symptoms_text', 'subspecialties_text', 'description', 'about_doctor']
df[text_cols] = df[text_cols].fillna('')

# ── BUILD SEARCH CORPUS ────────────────────────────────────────────────────
# We concatenate all medically relevant text per doctor.
# Speciality and symptoms_text carry the most signal; we weight them by
# repeating them once (simple but effective without any training).
#
# Format: "<specialty> | <symptoms> | <subspecialties> | <description>"
# The pipe separator helps the model treat each section as a distinct unit.
df['search_text'] = (
    df['Speciality'] + ' | ' +
    df['symptoms_text'] + ' | ' +
    df['subspecialties_text'] + ' | ' +
    df['description']
)

# Trim very long texts — MiniLM has a 256-token limit; truncating at 512 chars
# is a safe proxy without tokenizing (most doctor profiles fit anyway)
df['search_text'] = df['search_text'].str[:512]

# Drop doctors with no useful text at all (edge case)
df = df[df['search_text'].str.strip().str.len() > 10].reset_index(drop=True)


Loaded 17,119 doctors across 37 specialties.


## Embed doctors & build FAISS index

In [ ]:
# Load the sentence transformer model
print(f'Loading model: {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME)
EMBEDDING_DIM = model.get_sentence_embedding_dimension()
print(f'Model loaded. Embedding dimension: {EMBEDDING_DIM}')

if os.path.exists(INDEX_PATH):
    # ── LOAD EXISTING INDEX ───────────────────────────────────────────────
    print(f'\nFound saved index at "{INDEX_PATH}". Loading...')
    index = faiss.read_index(INDEX_PATH)
    print(f'Index loaded. {index.ntotal:,} vectors.')

else:
    # ── BUILD INDEX FROM SCRATCH ──────────────────────────────────────────
    print(f'\nNo saved index found. Embedding {len(df):,} doctors...')
    t0 = time.time()

    corpus = df['search_text'].tolist()

    embeddings = model.encode(
        corpus,
        batch_size=EMBEDDING_BATCH,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # L2-normalize → cosine similarity via dot product
    )

    elapsed = time.time() - t0
    print(f'\nEmbedding done in {elapsed:.1f}s. Shape: {embeddings.shape}')

    # ── BUILD FAISS FLAT INDEX (exact cosine search) ──────────────────────
    # IndexFlatIP = inner product (= cosine when vectors are L2-normalized)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(embeddings.astype('float32'))
    print(f'FAISS index built. {index.ntotal:,} vectors stored.')

    # Save to disk so future runs load instantly
    faiss.write_index(index, INDEX_PATH)
    print(f'Index saved to "{INDEX_PATH}".')

Loading model: all-MiniLM-L6-v2 ...


c:\Users\LENOVO\.conda\envs\Vezeeta\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 41120.63it/s]
BertModel LOAD REPORT f

Model loaded. Embedding dimension: 384

No saved index found. Embedding 17,119 doctors...


Batches: 100%|██████████| 67/67 [02:47<00:00,  2.50s/it]



Embedding done in 168.0s. Shape: (17119, 384)
FAISS index built. 17,119 vectors stored.
Index saved to "vezeeta_faiss.index".


## Search function

In [15]:
def search_doctors(query: str, top_k: int = TOP_K,
                   max_fee: int = None,
                   address: str = None,
                   specialty: str = None) -> pd.DataFrame:

    # 1. Embed the query (same model, same normalization)
    query_vec = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')

    # 2. Retrieve more candidates than needed so post-filters don't empty results
    fetch_k = min(top_k * 10, index.ntotal)
    scores, indices = index.search(query_vec, fetch_k)

    # 3. Build results dataframe
    result_df = df.iloc[indices[0]].copy()
    result_df['similarity_score'] = scores[0]

    # 4. Apply optional post-filters
    if max_fee is not None:
        result_df = result_df[result_df['fee'] <= max_fee]
    if address is not None:
        result_df = result_df[
            result_df['address'].str.lower().str.contains(address.lower())
        ]
    if specialty is not None:
        result_df = result_df[
            result_df['Speciality'].str.lower().str.contains(specialty.lower())
        ]

    # 5. Return top_k after filtering
    return result_df.head(top_k)[[
        'name', 'Speciality', 'address', 'fee',
        'waiting_time_min', 'reviews_count',
        'similarity_score', 'profile_url'
    ]].reset_index(drop=True)


In [39]:
results = search_doctors(
    query    = "I have chest pain and shortness of breath",
    top_k    = 5,
    max_fee  = None,     
    address  = 'Heliopolis',    
    specialty= None     
)

pd.set_option('display.max_colwidth', 30)
display(results)

,name,Speciality,address,fee,waiting_time_min,reviews_count,similarity_score,profile_url
0,Samir Maher,Chest and Respiratory,Heliopolis,800,-1,0,0.600643,https://www.vezeeta.com/en...
1,Taghreed Saeed Farag,Chest and Respiratory,Heliopolis,650,-1,0,0.597364,https://www.vezeeta.com/en...
2,Rajy Ghali,Chest and Respiratory,Heliopolis,600,17,77,0.595248,https://www.vezeeta.com/en...
3,Noha Osman,Chest and Respiratory,Heliopolis,350,42,798,0.592971,https://www.vezeeta.com/en...
4,Noha Osman,Chest and Respiratory,Heliopolis,350,42,798,0.592971,https://www.vezeeta.com/en...


In [27]:
test_queries = [
    "my child has fever and keeps vomiting",
    "I want to whiten my teeth and fix my smile",
    "severe back pain after gym workout",
    "feeling anxious and can't sleep at night",
    "acne and oily skin problem",
    "I've been trying to get pregnant for 2 years",
]

for q in test_queries:
    res = search_doctors(q, top_k=3)
    print(f"\nQuery: '{q}'")
    print(res[['name', 'Speciality', 'address', 'fee', 'similarity_score']].to_string(index=False))
    print('-' * 80)


Query: 'my child has fever and keeps vomiting'
            name Speciality        address  fee  similarity_score
Ramy Ahmed Foaad Pediatrics    Camp Caesar  350          0.638304
   Hassan Shawky Pediatrics  El-Obour City  300          0.628078
   Amani Youssef Pediatrics El-Fayoum City  420          0.619199
--------------------------------------------------------------------------------

Query: 'I want to whiten my teeth and fix my smile'
                       name Speciality   address  fee  similarity_score
                 Omar Gwely  Dentistry New Cairo  300          0.578996
smiley dental facial clinic  Dentistry    Faisal  200          0.539321
                  Rola Ramy  Dentistry  Sporting  385          0.508451
--------------------------------------------------------------------------------

Query: 'severe back pain after gym workout'
             name                       Speciality        address  fee  similarity_score
Makarious Shafter Physiotherapy and Sport Injuries 

## Scoring: blend similarity + reviews 

Pure cosine similarity ranks by relevance but ignores doctor quality signals.
This cell adds a simple re-ranking that blends similarity score + normalized review count.

In [ ]:
def search_doctors_ranked(query: str, top_k: int = TOP_K,
                          sim_weight: float = 0.7,
                          review_weight: float = 0.3,
                          **filters) -> pd.DataFrame:
    """
    Extended search with blended ranking:
      final_score = sim_weight * similarity + review_weight * normalized_reviews

    Tune sim_weight / review_weight to shift between relevance and popularity.
    """
    # Get a larger candidate pool from semantic search
    query_vec = model.encode([query], normalize_embeddings=True,
                             convert_to_numpy=True).astype('float32')
    fetch_k = min(top_k * 20, index.ntotal)
    scores, indices = index.search(query_vec, fetch_k)

    candidates = df.iloc[indices[0]].copy()
    candidates['similarity_score'] = scores[0]

    # Normalize review count to [0, 1] within the candidate pool
    max_reviews = candidates['reviews_count'].max() or 1
    candidates['review_score'] = candidates['reviews_count'] / max_reviews

    # Blended final score
    candidates['final_score'] = (
        sim_weight    * candidates['similarity_score'] +
        review_weight * candidates['review_score']
    )

    # Apply filters from **filters kwargs (same keys as search_doctors)
    if filters.get('max_fee'):
        candidates = candidates[candidates['fee'] <= filters['max_fee']]
    if filters.get('city'):
        candidates = candidates[
            candidates['address'].str.lower().str.contains(filters['city'].lower())
        ]
    if filters.get('specialty'):
        candidates = candidates[
            candidates['Speciality'].str.lower().str.contains(filters['specialty'].lower())
        ]

    return candidates.sort_values('final_score', ascending=False).head(top_k)[[
        'name', 'Speciality', 'address', 'fee', 'reviews_count',
        'waiting_time_min', 'similarity_score', 'review_score',
        'final_score', 'profile_url'
    ]].reset_index(drop=True)


results_ranked = search_doctors_ranked(
    query         = "chest pain and shortness of breath",
    top_k         = 5,
    sim_weight    = 0.7,
    review_weight = 0.3,
    max_fee       = 800
)
display(results_ranked)

,name,Speciality,address,fee,reviews_count,waiting_time_min,similarity_score,review_score,final_score,profile_url
0,Mohame...,Chest ...,El-Haram,600,1980,23,0.585540,1.000000,0.709878,https:...
1,Ahmed ...,Chest ...,6Th Of...,500,1000,41,0.583645,0.505051,0.560067,https:...
2,Noha O...,Chest ...,Heliop...,350,798,42,0.597005,0.403030,0.538812,https:...
3,Noha O...,Chest ...,Heliop...,350,798,42,0.597005,0.403030,0.538812,https:...
4,Haytha...,Chest ...,Sidy B...,350,812,60,0.575832,0.410101,0.526113,https:...


## Evaluate quality

Without ground-truth labels, we do a manual specialty precision check:
given a symptom query with an obvious expected specialty, how often do top-K results match?

In [32]:
# Each entry: (query, expected_specialty)
eval_cases = [
    ("chest pain and irregular heartbeat",     "Cardiology"),
    ("toothache and bleeding gums",             "Dentistry"),
    ("child has high fever and diarrhea",       "Pediatrics"),
    ("back pain after sports injury",           "Orthopedics"),
    ("acne and skin rash",                      "Dermatology"),
    ("feeling depressed and anxious",           "Psychiatry"),
    ("difficulty getting pregnant",             "Reproductive Medicine & Infertility"),
    ("blurry vision and eye strain",            "Ophthalmology"),
    ("ear pain and hearing loss",               "ENT"),
    ("frequent urination and kidney pain",      "Urology"),
]

print(f"{'Query':<45} {'Expected':<35} Top-3 Specialties")
print('-' * 110)

hits_at_1 = 0
hits_at_3 = 0

for query, expected in eval_cases:
    res = search_doctors(query, top_k=3)
    top3_specs = res['Speciality'].tolist()
    hit1 = expected.lower() in top3_specs[0].lower() if top3_specs else False
    hit3 = any(expected.lower() in s.lower() for s in top3_specs)
    hits_at_1 += hit1
    hits_at_3 += hit3
    tag = '✓' if hit3 else '✗'
    print(f"{tag} {query:<44} {expected:<35} {' | '.join(top3_specs)}")

n = len(eval_cases)
print(f'\nPrecision@1: {hits_at_1}/{n} = {hits_at_1/n:.0%}')
print(f'Precision@3: {hits_at_3}/{n} = {hits_at_3/n:.0%}')


Query                                         Expected                            Top-3 Specialties
--------------------------------------------------------------------------------------------------------------
✓ chest pain and irregular heartbeat           Cardiology                          Cardiology | Cardiology | Cardiology
✓ toothache and bleeding gums                  Dentistry                           Dentistry | Dentistry | Dentistry
✓ child has high fever and diarrhea            Pediatrics                          Pediatrics | Pediatrics | Pediatrics
✗ back pain after sports injury                Orthopedics                         Physiotherapy and Sport Injuries | Physiotherapy and Sport Injuries | Physiotherapy and Sport Injuries
✓ acne and skin rash                           Dermatology                         Dermatology | Dermatology | Dermatology
✓ feeling depressed and anxious                Psychiatry                          Psychiatry | Psychiatry | Psychiatry
✓ d

Runs **26 queries** across **26 specialties** and reports:

| Metric | What it tells you |
|--------|-------------------|
| **Precision@K** | Does the correct specialty appear in the top-K results? |
| **MRR** | On average, how early does the correct specialty appear? (1.0 = always #1) |
| **Per-specialty table** | Which specialties the model handles well vs. struggles with |
| **Misses** | Queries where the correct specialty wasn't found in top-5 — actionable debug output |

In [34]:
EVAL_CASES = [
    ("chest pain, palpitations and irregular heartbeat",       "Cardiology"),
    ("toothache, bleeding gums and root canal pain",           "Dentistry"),
    ("child has high fever, vomiting and diarrhea",            "Pediatrics"),
    ("knee pain and fracture after accident",                  "Orthopedics"),
    ("acne, oily skin and hair loss",                          "Dermatology"),
    ("feeling depressed, anxious and cannot sleep",            "Psychiatry"),
    ("difficulty getting pregnant after 2 years of trying",   "Reproductive Medicine & Infertility"),
    ("blurry vision, eye strain and cataract",                 "Ophthalmology"),
    ("ear pain, hearing loss and tonsillitis",                 "ENT"),
    ("frequent urination, kidney stones and prostate pain",    "Urology"),
    ("shortness of breath, asthma and chronic cough",          "Chest and Respiratory"),
    ("stomach pain, bloating and irritable bowel",             "Gastroenterology"),
    ("shoulder physiotherapy after sports injury",             "Physiotherapy and Sport Injuries"),
    ("obesity management and weight loss surgery",             "Obesity and Laparoscopic Surgery"),
    ("diabetes management and thyroid disorder",               "Diabetes and Endocrinology"),
    ("joint pain, arthritis and lupus",                        "Rheumatology"),
    ("memory loss, headache and epilepsy",                     "Neurology"),
    ("abdominal hernia and appendicitis",                      "General Surgery"),
    ("healthy diet plan and nutritional deficiency",           "Dietitian and Nutrition"),
    ("chemotherapy side effects and tumor follow-up",          "Oncology"),
    ("varicose veins and vascular blockage",                   "Vascular Surgery"),
    ("kidney failure and dialysis",                            "Nephrology"),
    ("nose job and liposuction consultation",                  "Plastic Surgery"),
    ("liver disease and hepatitis B follow-up",                "Hepatology"),
    ("blood disorder and anemia treatment",                    "Hematology"),
    ("chronic back pain management and nerve block",           "Pain Management"),
]

K_VALUES = [1, 3, 5, 10]   # evaluate Precision@1, @3, @5, @10

# ── RUN EVALUATION ───────────────────────────────────────────────────────────
print("Running evaluation on", len(EVAL_CASES), "queries...\n")

rows = []   # one row per query

for query, expected in EVAL_CASES:
    # Fetch top-10 results (max K we evaluate)
    res = search_doctors(query, top_k=10)
    predicted_specs = res['Speciality'].tolist()

    # Reciprocal Rank — find first hit
    rr = 0.0
    for rank, spec in enumerate(predicted_specs, start=1):
        if expected.lower() in spec.lower():
            rr = 1.0 / rank
            break

    # Hit@K for each K value
    hits = {}
    for k in K_VALUES:
        top_k_specs = predicted_specs[:k]
        hits[k] = int(any(expected.lower() in s.lower() for s in top_k_specs))

    rows.append({
        'query':    query,
        'expected': expected,
        'top_pred': predicted_specs[0] if predicted_specs else '—',
        'RR':       rr,
        **{f'hit@{k}': hits[k] for k in K_VALUES},
        'top5_specialties': ' | '.join(predicted_specs[:5]),
    })

results_df = pd.DataFrame(rows)

# ── OVERALL METRICS ──────────────────────────────────────────────────────────
mrr = results_df['RR'].mean()

print("=" * 65)
print("  OVERALL EVALUATION RESULTS")
print("=" * 65)
print(f"  Queries evaluated : {len(results_df)}")
print(f"  MRR               : {mrr:.3f}  (1.0 = always rank #1)")
for k in K_VALUES:
    p_at_k = results_df[f'hit@{k}'].mean()
    bar = '█' * int(p_at_k * 20) + '░' * (20 - int(p_at_k * 20))
    print(f"  Precision@{k:<2}      : {p_at_k:.2%}  {bar}")
print("=" * 65)

# ── PER-QUERY BREAKDOWN ──────────────────────────────────────────────────────
print("\n── Per-query breakdown (sorted by Reciprocal Rank) ──\n")

display_df = results_df[['query', 'expected', 'top_pred', 'RR', 'hit@1', 'hit@3', 'hit@5']].copy()
display_df['RR']     = display_df['RR'].map(lambda x: f"{x:.2f}")
display_df['hit@1']  = display_df['hit@1'].map(lambda x: '✓' if x else '✗')
display_df['hit@3']  = display_df['hit@3'].map(lambda x: '✓' if x else '✗')
display_df['hit@5']  = display_df['hit@5'].map(lambda x: '✓' if x else '✗')
display_df = display_df.sort_values('RR', ascending=True)

pd.set_option('display.max_colwidth', 42)
pd.set_option('display.width', 200)
display(display_df.reset_index(drop=True))

# ── PER-SPECIALTY METRICS ────────────────────────────────────────────────────
print("\n── Per-specialty performance (MRR, sorted best → worst) ──\n")

specialty_stats = (
    results_df
    .groupby('expected')
    .agg(
        MRR        = ('RR',    'mean'),
        Hit_at_1   = ('hit@1', 'mean'),
        Hit_at_5   = ('hit@5', 'mean'),
        N_queries  = ('RR',    'count'),
    )
    .sort_values('MRR', ascending=False)
    .reset_index()
)
specialty_stats.columns = ['Specialty', 'MRR', 'Hit@1', 'Hit@5', 'N_queries']
specialty_stats['MRR']   = specialty_stats['MRR'].map(lambda x: f"{x:.2f}")
specialty_stats['Hit@1'] = specialty_stats['Hit@1'].map(lambda x: f"{x:.0%}")
specialty_stats['Hit@5'] = specialty_stats['Hit@5'].map(lambda x: f"{x:.0%}")

pd.set_option('display.max_colwidth', 40)
display(specialty_stats)

# ── MISSES ───────────────────────────────────────────────────────────────────
misses = results_df[results_df['hit@5'] == 0]
if len(misses):
    print(f"\n── Misses (not found in top-5) — needs investigation ──\n")
    for _, row in misses.iterrows():
        print(f"  Query    : {row['query']}")
        print(f"  Expected : {row['expected']}")
        print(f"  Got      : {row['top5_specialties']}")
        print()
else:
    print("\n✓ No misses — correct specialty found in top-5 for all queries.")

# ── INTERPRETATION GUIDE ─────────────────────────────────────────────────────
print("""
── How to interpret these results ──────────────────────────────────────

  MRR ≥ 0.80  →  Excellent. Correct specialty is almost always #1 or #2.
  MRR 0.60-0.79 →  Good. Correct specialty is within top-2 most of the time.
  MRR 0.40-0.59 →  Fair. Acceptable for a cold-start system; tune corpus text.
  MRR < 0.40  →  Needs work. Check if that specialty has sparse symptom data.

""")

Running evaluation on 26 queries...

  OVERALL EVALUATION RESULTS
  Queries evaluated : 26
  MRR               : 0.962  (1.0 = always rank #1)
  Precision@1       : 92.31%  ██████████████████░░
  Precision@3       : 100.00%  ████████████████████
  Precision@5       : 100.00%  ████████████████████
  Precision@10      : 100.00%  ████████████████████

── Per-query breakdown (sorted by Reciprocal Rank) ──



,query,expected,top_pred,RR,hit@1,hit@3,hit@5
0,nose job and liposuction consultation,Plastic Surgery,ENT,0.50,✗,✓,✓
1,blood disorder and anemia treatment,Hematology,Pediatrics,0.50,✗,✓,✓
2,"chest pain, palpitations and irregular...",Cardiology,Cardiology,1.00,✓,✓,✓
3,liver disease and hepatitis B follow-up,Hepatology,Hepatology,1.00,✓,✓,✓
4,kidney failure and dialysis,Nephrology,Nephrology,1.00,✓,✓,✓
5,varicose veins and vascular blockage,Vascular Surgery,Vascular Surgery,1.00,✓,✓,✓
6,chemotherapy side effects and tumor fo...,Oncology,Oncology,1.00,✓,✓,✓
7,healthy diet plan and nutritional defi...,Dietitian and Nutrition,Dietitian and Nutrition,1.00,✓,✓,✓
8,abdominal hernia and appendicitis,General Surgery,General Surgery,1.00,✓,✓,✓
9,"memory loss, headache and epilepsy",Neurology,Neurology,1.00,✓,✓,✓



── Per-specialty performance (MRR, sorted best → worst) ──



,Specialty,MRR,Hit@1,Hit@5,N_queries
0,Cardiology,1.00,100%,100%,1
1,Chest and Respiratory,1.00,100%,100%,1
2,Urology,1.00,100%,100%,1
3,Rheumatology,1.00,100%,100%,1
4,Reproductive Medicine & Infertility,1.00,100%,100%,1
5,Psychiatry,1.00,100%,100%,1
6,Physiotherapy and Sport Injuries,1.00,100%,100%,1
7,Pediatrics,1.00,100%,100%,1
8,Pain Management,1.00,100%,100%,1
9,Orthopedics,1.00,100%,100%,1



✓ No misses — correct specialty found in top-5 for all queries.

── How to interpret these results ──────────────────────────────────────

  MRR ≥ 0.80  →  Excellent. Correct specialty is almost always #1 or #2.
  MRR 0.60-0.79 →  Good. Correct specialty is within top-2 most of the time.
  MRR 0.40-0.59 →  Fair. Acceptable for a cold-start system; tune corpus text.
  MRR < 0.40  →  Needs work. Check if that specialty has sparse symptom data.


